# v51 — KTF³ Filament Spin Parity Test

**KTF³ Analysis — Andrew Cotting, April 2026**

## Hypothesis

The Klein bottle topology φ(x,y,z) = (−x, −y, −z, t+T) implies a **parity flip** along the KTF³ axis (l=277°, b=−31°).  
For cosmic filaments, this predicts:

- Filaments **parallel** to the KTF³ axis → **anti-correlated** spin on opposite sides
- Filaments **perpendicular** to the KTF³ axis → **correlated** spin (control)

## Data

**Carrón Duque et al. (2022)** SDSS filament catalog (public, VizieR J/A+A/665/A44).  
~17,000 filaments up to z=0.15 with galaxy membership and spin proxies from SDSS spectroscopy.

## Null hypothesis

No preferred spin orientation relative to the KTF³ axis. Galaxy velocity asymmetry is isotropic.  
Null tested via: (1) random axis control, (2) redshift shuffle of galaxy assignments.

## Status note

- v51a: synthetic data (pipeline test) → σ=0.33 NULL (expected, not real data)
- v51 (this notebook): real SDSS data via astroquery
- v51b: awaiting Oxford/MIGHTEE filament data (Dr. Jung, outreach pending)

In [ ]:
# === INSTALLATION ===
!pip install numpy scipy matplotlib astropy astroquery healpy -q
print('Packages ready.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import healpy as hp
import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.vizier import Vizier
import warnings
warnings.filterwarnings('ignore')

# KTF3 axis (galactic coordinates)
KTF3_L = 277.0  # degrees
KTF3_B = -31.0  # degrees

print(f'KTF³ axis: l={KTF3_L}°, b={KTF3_B}°')

# Convert KTF3 axis to unit vector
ktf3_coord = SkyCoord(l=KTF3_L*u.deg, b=KTF3_B*u.deg, frame='galactic')
ktf3_icrs = ktf3_coord.icrs
ktf3_vec = np.array([
    np.cos(ktf3_icrs.dec.rad) * np.cos(ktf3_icrs.ra.rad),
    np.cos(ktf3_icrs.dec.rad) * np.sin(ktf3_icrs.ra.rad),
    np.sin(ktf3_icrs.dec.rad)
])
print(f'KTF³ axis vector (ICRS): {ktf3_vec.round(3)}')

In [ ]:
# === DOWNLOAD: Carron Duque+2022 SDSS Filament Catalog ===
# VizieR catalog: J/A+A/665/A44
# Contains: filament ID, RA, Dec, z, length, galaxy members

print('Downloading Carron Duque+2022 filament catalog...')
Vizier.ROW_LIMIT = -1

try:
    catalogs = Vizier.get_catalogs('J/A+A/665/A44')
    print(f'Tables found: {list(catalogs.keys())}')
    for name, tab in catalogs.items():
        print(f'  {name}: {len(tab)} rows, columns: {tab.colnames[:8]}')
except Exception as e:
    print(f'VizieR download failed: {e}')
    print('Trying alternative: Tempel+2014 filament catalog (J/A+A/566/A1)')
    try:
        catalogs = Vizier.get_catalogs('J/A+A/566/A1')
        print(f'Tempel+2014 tables: {list(catalogs.keys())}')
    except Exception as e2:
        print(f'Also failed: {e2}')

In [ ]:
# === DOWNLOAD: SDSS Galaxy Velocities for Spin Proxy ===
# We use galaxy peculiar velocities projected perpendicular to the filament spine
# as a proxy for filament spin (angular momentum).
#
# Source: SDSS DR16 spectroscopic galaxies (z < 0.15)
# Velocity proxy: v_proj = v_los * sin(angle_to_filament_axis)

print('Downloading SDSS DR16 galaxy sample...')
from astroquery.sdss import SDSS

query = """
SELECT TOP 50000
  p.objid, p.ra, p.dec,
  s.z, s.velDisp,
  s.zErr
FROM PhotoObj AS p
JOIN SpecObj AS s ON s.bestobjid = p.objid
WHERE s.class = 'GALAXY'
  AND s.z BETWEEN 0.01 AND 0.15
  AND s.zWarning = 0
  AND s.zErr < 0.001
"""

try:
    gals = SDSS.query_sql(query)
    print(f'SDSS galaxies downloaded: {len(gals)} rows')
    print(gals[:3])
except Exception as e:
    print(f'SDSS query failed: {e}')
    print('Will use synthetic data for pipeline test.')
    gals = None

In [ ]:
# === SYNTHETIC FALLBACK (if real data unavailable) ===
# Generates mock filaments with known spin signal for pipeline validation.
# NOT for scientific conclusions.

USE_SYNTHETIC = (gals is None)

if USE_SYNTHETIC:
    print('⚠️  Using SYNTHETIC data — pipeline test only, no scientific conclusions.')
    np.random.seed(42)
    N_fil = 500
    # Random filament positions
    ra_fil  = np.random.uniform(100, 260, N_fil)
    dec_fil = np.random.uniform(-10, 60, N_fil)
    z_fil   = np.random.uniform(0.02, 0.12, N_fil)
    # Random filament orientation angles (degrees, 0=N, 90=E)
    pa_fil  = np.random.uniform(0, 180, N_fil)
    # Spin signal: random (null simulation)
    spin_fil = np.random.normal(0, 1, N_fil)
    lengths  = np.random.uniform(5, 50, N_fil)  # Mpc
    print(f'Synthetic: {N_fil} filaments generated.')
else:
    print('✅ Using real SDSS data.')

In [ ]:
# === GEOMETRY: Angle between filament and KTF3 axis ===
# For each filament, compute the angle theta between its spine direction
# and the KTF3 preferred axis.
# theta < 45° → parallel; theta > 45° → perpendicular

def filament_ktf3_angle(ra, dec, pa):
    """
    Compute angle between filament orientation and KTF3 axis.
    ra, dec: filament center (degrees)
    pa: position angle of filament spine (degrees, N through E)
    Returns: angle in degrees [0, 90]
    """
    # Filament direction vector in 3D
    ra_rad  = np.radians(ra)
    dec_rad = np.radians(dec)
    pa_rad  = np.radians(pa)
    
    # Local N and E unit vectors at filament position
    e_north = np.array([
        -np.sin(dec_rad)*np.cos(ra_rad),
        -np.sin(dec_rad)*np.sin(ra_rad),
         np.cos(dec_rad)
    ])
    e_east = np.array([-np.sin(ra_rad), np.cos(ra_rad), 0.0])
    
    # Filament direction
    fil_vec = np.cos(pa_rad)*e_north + np.sin(pa_rad)*e_east
    
    # Angle with KTF3 axis
    cos_angle = np.clip(np.dot(fil_vec, ktf3_vec), -1, 1)
    angle = np.degrees(np.arccos(np.abs(cos_angle)))  # [0,90]
    return angle

if USE_SYNTHETIC:
    angles = np.array([filament_ktf3_angle(ra_fil[i], dec_fil[i], pa_fil[i])
                       for i in range(N_fil)])
    print(f'Angles computed. Mean={angles.mean():.1f}°, std={angles.std():.1f}°')
    parallel_mask     = angles < 45
    perpendicular_mask = angles >= 45
    print(f'Parallel filaments (θ<45°): {parallel_mask.sum()}')
    print(f'Perpendicular (θ≥45°):      {perpendicular_mask.sum()}')

In [ ]:
# === STATISTICAL TEST ===
# KTF3 prediction: parallel filaments show ANTI-correlated spin (parity flip)
# → mean spin for parallel filaments should differ from perpendicular
# Rayleigh test + mean spin comparison

N_MC = 5000

if USE_SYNTHETIC:
    spin_parallel = spin_fil[parallel_mask]
    spin_perp     = spin_fil[perpendicular_mask]
    
    # Observed statistic: difference in mean |spin|
    T_obs = np.abs(spin_parallel.mean()) - np.abs(spin_perp.mean())
    print(f'Mean spin (parallel):     {spin_parallel.mean():.4f} ± {spin_parallel.std()/np.sqrt(len(spin_parallel)):.4f}')
    print(f'Mean spin (perpendicular): {spin_perp.mean():.4f} ± {spin_perp.std()/np.sqrt(len(spin_perp)):.4f}')
    print(f'T_obs = {T_obs:.4f}')
    
    # Monte Carlo null: shuffle axis assignment
    T_mc = np.zeros(N_MC)
    for i in range(N_MC):
        idx = np.random.permutation(N_fil)
        ang_shuffled = angles[idx]
        par = spin_fil[ang_shuffled < 45]
        per = spin_fil[ang_shuffled >= 45]
        T_mc[i] = np.abs(par.mean()) - np.abs(per.mean()) if (len(par)>0 and len(per)>0) else 0
    
    p_value = np.mean(np.abs(T_mc) >= np.abs(T_obs))
    sigma = (T_obs - T_mc.mean()) / T_mc.std() if T_mc.std() > 0 else 0
    print(f'\nNull MC (N={N_MC}): mean={T_mc.mean():.4f}, std={T_mc.std():.4f}')
    print(f'σ = {sigma:.3f}')
    print(f'p = {p_value:.4f}')
    print()
    if USE_SYNTHETIC:
        print('⚠️  SYNTHETIC DATA — result not scientifically meaningful.')
        print('    Expected: σ ≈ 0 (null by construction)')

In [ ]:
# === PLOT ===
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('v51 — KTF³ Filament Spin Parity Test', fontsize=13, fontweight='bold')

if USE_SYNTHETIC:
    # Panel 1: Angle distribution
    ax = axes[0]
    ax.hist(angles, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(45, color='red', ls='--', label='θ = 45° (boundary)')
    ax.set_xlabel('Angle to KTF³ axis (degrees)')
    ax.set_ylabel('Count')
    ax.set_title('Filament–KTF³ Angle Distribution')
    ax.legend()

    # Panel 2: Spin distribution by group
    ax = axes[1]
    ax.hist(spin_fil[parallel_mask], bins=25, alpha=0.6, color='crimson',
            label=f'Parallel (θ<45°, N={parallel_mask.sum()})')
    ax.hist(spin_fil[perpendicular_mask], bins=25, alpha=0.6, color='royalblue',
            label=f'Perpendicular (θ≥45°, N={perpendicular_mask.sum()})')
    ax.set_xlabel('Spin proxy')
    ax.set_ylabel('Count')
    ax.set_title('Spin Distribution by Orientation')
    ax.legend()

    # Panel 3: MC null distribution
    ax = axes[2]
    ax.hist(T_mc, bins=50, color='gray', edgecolor='white', alpha=0.8, label='MC null')
    ax.axvline(T_obs, color='red', lw=2, label=f'T_obs = {T_obs:.3f}')
    ax.set_xlabel('Test statistic T')
    ax.set_ylabel('Count')
    ax.set_title(f'MC Null Distribution (σ={sigma:.2f})')
    ax.legend()

plt.tight_layout()
plt.savefig('v51_filament_spin.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved: v51_filament_spin.png')

In [ ]:
# === RESULT SUMMARY ===
print('=' * 55)
print('v51 — KTF³ Filament Spin Parity Test')
print('=' * 55)
if USE_SYNTHETIC:
    print('Data:    SYNTHETIC (pipeline validation only)')
else:
    print('Data:    SDSS DR16 + Carron Duque+2022 filaments')
print(f'N_fil:   {N_fil}')
if USE_SYNTHETIC:
    print(f'σ:       {sigma:.3f}   [SYNTHETIC — not meaningful]')
    print(f'p:       {p_value:.4f}')
    print()
    print('Verdict: Awaiting real SDSS data (pipeline OK)')
    print()
    print('Next steps:')
    print('  1. Run with real Carron Duque catalog (VizieR J/A+A/665/A44)')
    print('  2. v51b: await Oxford/MIGHTEE data (Dr. Jung outreach pending)')
    print()
    print('KTF³ prediction: σ > 2 for parallel vs perpendicular spin asymmetry')
print('=' * 55)